### Logistic Regression: Week 4
#### Step 1: Load and Check the data

In [179]:
import pandas as pd

df = pd.read_csv("datasets/train.csv")

print(df.head())
print(df.info())
print(df.describe())
print(df["Embarked"].head(20))

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
<c

#### Step 2: fill Null values

In [180]:
# Fill age with median
df["Age"] = df["Age"].fillna(df["Age"].median())

# Fill Embarked with Mode
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Drop Cabin
df = df.drop(columns=["Cabin"])

#### Step 3: Feature Engineering

Phase 3 needs Titles and other features extracted before dropping uneeded columns

In [181]:
# 1️ Extract Title FIRST
df["Title"] = df["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)

# Group rare titles
rare_titles = ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona']
df["Title"] = df["Title"].replace(rare_titles, "Other")

# 2️ Create FamilySize
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# 3️ Create IsAlone
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

# 4️⃣ NOW drop unnecessary columns
df = df.drop(columns=["PassengerId", "Name", "Ticket"])

#### Step 4: One-Hot Encode

In [182]:
df = pd.get_dummies(df, drop_first=True)

#### Step 5: Train-Test Split

In [183]:
from sklearn.model_selection import train_test_split

X = df.drop("Survived", axis=1)
y = df["Survived"]

X_train,  X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#### Step 6: Scale (Important for Logisitic Regression)

In [184]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#### Step 7: Logitistic Regression

In [185]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000)

log_reg.fit(X_train_scaled, y_train)

y_pred = log_reg.predict(X_test_scaled)

#### Step 8: Evaluate

In [186]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8156424581005587
[[89 16]
 [17 57]]
              precision    recall  f1-score   support

           0       0.84      0.85      0.84       105
           1       0.78      0.77      0.78        74

    accuracy                           0.82       179
   macro avg       0.81      0.81      0.81       179
weighted avg       0.82      0.82      0.82       179



### Part 2
#### Step 1: Tune Regulatization with `LogisticRegressionCV`
Before we were doing default regularization, now we will let the model choose the best C automatically.

What this does:
* C is Inverse regularization strength:
    * Small C → Stronger regularization
    * Large C → Weaker regularization

What is C?:
* `Cs=[0.001, 0.01, 0.1, 1, 10, 100]`
* In Logistic regerssion `C = inverse of regularization strength`
    * C=1/λ​
    * Where:
        * `λ​ = regularization strength`
        * `C = how much freedom the model has`

What is CV?:
* CV is Cross-validation
* `cv = 5` splits my training data into 5 parts:
    * Train on 4 folds
    * Validate on 1 fold
    * Repeat 5 times
    * Average performance
    * Pick best C
* So instead of guessing regularization strength, the model chooses it statistically.

What is `penalty = "l2"`
* This defines the type of regulariation
    * L2 penalty adds `λ ∑ w²`
* What does L2 do?
    * Shrinks coefficients toward zero
    * Keeps all features
    * Reduces variance
    * Smooths model
* It does NOT eliminate feautres

Alternatively: "l1":
* This adds : `λ ∑ ∣w∣`
* L1:
    * Can shrink some coefficients exactly to 0
    performs feature selection
    Produces sparse models

What is `solver = "lbfgs"`:
* The solver is the optimization algorithim
* Logistic regression is solved via numerical optimization.
* `lbgs`:
    * Stands for: `Limited-Memory Boryden-Fletcher-Goldfarb-Shano`
    * Efficient for medium-sized datasets
    * Supports L2
    * Default for sklearn

In [187]:
from sklearn.linear_model import LogisticRegressionCV

log_cv = LogisticRegressionCV(
    Cs=10,
    cv=5,
    penalty="l2",
    solver="lbfgs",
    max_iter=5000,
    random_state=42
)

log_cv.fit(X_train_scaled, y_train)

y_pred_cv = log_cv.predict(X_test)
log_cv_acc = accuracy_score(y_test, y_pred_cv)
best_c = log_cv.C_[0]

print("Accuracy:", accuracy_score(y_test, y_pred_cv))
print("Best C:", log_cv.C_[0])
print(confusion_matrix(y_test, y_pred_cv))
print(classification_report(y_test, y_pred_cv))

Accuracy: 0.6927374301675978
Best C: 2.782559402207126
[[100   5]
 [ 50  24]]
              precision    recall  f1-score   support

           0       0.67      0.95      0.78       105
           1       0.83      0.32      0.47        74

    accuracy                           0.69       179
   macro avg       0.75      0.64      0.63       179
weighted avg       0.73      0.69      0.65       179



c:\Users\asume\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
c:\Users\asume\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
c:\Users\asume\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-le

#### Step 2: Compare Random Forest
Random forest is a non-linear model.

Differences:
* Logistic Regression has Linear decision boundaries
* Random forest is nonlinear, captures interactions automatically, and handles Mixed features naturally


In [188]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
rf_acc = accuracy_score(y_test, y_pred_rf)

print("RF Accuracy:", accuracy_score(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

RF Accuracy: 0.8268156424581006
[[89 16]
 [15 59]]
              precision    recall  f1-score   support

           0       0.86      0.85      0.85       105
           1       0.79      0.80      0.79        74

    accuracy                           0.83       179
   macro avg       0.82      0.82      0.82       179
weighted avg       0.83      0.83      0.83       179



#### Step 3: Feature Importance (Random forest)

In [189]:
feature_importance = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(feature_importance.head(10))

Fare          0.239390
Age           0.216127
Title_Mr      0.133511
Sex_male      0.113295
Pclass        0.078261
FamilySize    0.046972
Title_Mrs     0.043062
SibSp         0.031361
Title_Miss    0.026636
Parch         0.023100
dtype: float64


#### Step 4: Interpret Logistic Regression Coefficients

In [190]:
coef_df = pd.Series(
    log_cv.coef_[0],
    index=X_train.columns
).sort_values()

print(coef_df)

Title_Mr      -1.659647
Title_Miss    -0.956488
Sex_male      -0.866463
Pclass        -0.722641
Title_Other   -0.469236
Title_Mrs     -0.445944
SibSp         -0.421339
FamilySize    -0.378845
Age           -0.231012
Embarked_S    -0.202481
Parch         -0.179053
IsAlone       -0.159024
Embarked_Q    -0.048084
Title_Mme      0.072162
Title_Mlle     0.103391
Title_Ms       0.113900
Fare           0.209900
dtype: float64


In [191]:
# Final Comparison Output
print("=== Model Comparison ===")
print(f"LogisticRegressionCV Accuracy: {log_cv_acc:.4f}")
print(f"Best C (Regularization Strength): {best_c}")
print(f"RandomForest Accuracy: {rf_acc:.4f}")

=== Model Comparison ===
LogisticRegressionCV Accuracy: 0.6927
Best C (Regularization Strength): 2.782559402207126
RandomForest Accuracy: 0.8268


#### Insight:
* Linear models are great for interpretability, but require careful feature engineering (e.g., Titles, FamilySize, IsAlone).
* **RandomForests** handle complex relationships automatically
* Key Lessons in ML: Model choice matters and sometimes adding features is as important as tuning hyperparameters.

### Phase 3
#### Step 1: Feature Engineering (Before Splitting)

In [192]:
df = pd.read_csv("datasets/train.csv")

# Extract Title from Name
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Group rare titles into "Other"
rare_titles = ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona']
df['Title'] = df['Title'].replace(rare_titles, 'Other')

# Create FamilySize
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Create IsAlone
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Fill age with median
df["Age"] = df["Age"].fillna(df["Age"].median())

# Fill Embarked with Mode
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Drop unused columns
df = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

#### Step 2: Split Data

In [193]:
X = df.drop('Survived', axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#### Step 3: Encode Categorical Features

In [194]:
categorical_cols = ['Sex', 'Embarked', 'Title']
X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test  = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)


#### Step 4: Scale Numeric Features

In [195]:
numeric_cols = ['Age','Fare','FamilySize']
scaler = StandardScaler()

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])

#### Step 5 Train logisticRegressionCV Again

In [197]:
log_cv2 = LogisticRegressionCV(
    Cs=10,
    cv=5,
    penalty='l2',
    solver='lbfgs',
    max_iter=5000,
    random_state=42
)

log_cv2.fit(X_train, y_train)
y_pred = log_cv.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Best C:", log_cv2.C_[0])
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8156424581005587
Best C: 21.54434690031882
[[89 16]
 [17 57]]
              precision    recall  f1-score   support

           0       0.84      0.85      0.84       105
           1       0.78      0.77      0.78        74

    accuracy                           0.82       179
   macro avg       0.81      0.81      0.81       179
weighted avg       0.82      0.82      0.82       179



c:\Users\asume\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
c:\Users\asume\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
c:\Users\asume\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-le

# Week 4: Model Improvement & Robustness

## House Prices (Linear Regression)

**Steps Completed:**
- Feature engineering: combined/derived features (e.g., TotalLivingArea), handled nulls appropriately
- Scaling numerical features with `StandardScaler`
- Applied regularization with `RidgeCV` and `LassoCV`
- Added polynomial features to capture non-linear relationships
- Evaluated using cross-validation (RMSE, R²)
- Compared models: baseline vs regularized vs polynomial

**Results:**

| Model              | RMSE    | R²      |
|-------------------|---------|---------|
| Linear Regression  | 1.8686  | -17.7113 |
| RidgeCV            | 0.1640  | 0.8558  |
| LassoCV            | 0.1471  | 0.8840  |
| Polynomial RidgeCV | 0.2799  | 0.5801  |

**Takeaways:**
- Regularization improved predictive performance and reduced overfitting
- Polynomial features help capture non-linearity but can overfit if too complex
- Scaling ensures fair comparisons and stable optimization
- LassoCV can zero out less important features, performing feature selection automatically

---

## Titanic Survival (Logistic Regression)

**Steps Completed:**
- Feature engineering: extracted `Title` from `Name`, created `FamilySize` and `IsAlone`
- Dropped irrelevant columns: `PassengerId`, `Name`, `Ticket`, `Cabin`
- One-hot encoded categorical features (`Sex`, `Embarked`, `Title`) and aligned train/test sets
- Scaled numerical features (`Age`, `Fare`, `FamilySize`)
- Trained `LogisticRegressionCV` with cross-validated regularization strength
- Compared against Random Forest classifier

**Results:**

| Model                  | Accuracy | Notes |
|------------------------|---------|-------|
| LogisticRegressionCV   | ~0.6648 | Interpretable, C selected automatically |
| Random Forest          | ~0.8101 | Captures non-linear patterns, higher accuracy |

**Takeaways:**
- Feature engineering significantly boosts model performance
- Proper categorical encoding is crucial; string columns must be numeric
- LogisticRegressionCV automates hyperparameter tuning and keeps the model interpretable
- Random Forest provides better predictive accuracy at the cost of interpretability

---

## Key Concepts Learned

- Feature engineering is critical for improving model robustness
- Regularization (L1/L2) prevents overfitting and stabilizes coefficients
- Cross-validation provides more reliable evaluation than a single train-test split
- Scaling numerical features is essential for regularized linear/logistic models
- Pipeline order matters: missing values → feature engineering → drop raw columns → encoding → scaling → training → evaluation
- Evaluation metrics differ by task: regression → RMSE, R²; classification → Accuracy, Precision, Recall, F1, ROC-AUC